# Ensemble Learning to forecast weather

Ensemble has 2 parts: the weak learners and the method of combining these weak learners

There are also 3 types of ensemble learning:
- **Bagging**: each weak learner makes a decision, and the final decision from this ensemble is usually average of decisions or majority vote
- **Boosting**: each weak learner is trained sequentially, and each new model tries to correct the error of the previous one. By focusing on harder and harder to classify problems one weak learner after another, the weighted predictions makes the ensemble much stronger at predicting.
- **Stacking**: Each weak learner is independent, and then a meta-model (a higher level model) learns how to combine these predictions to make the final output. Multi-layer stacking is also possible: feeding first level of base models outputs into a second level and then into a meta-model to create a final output. This can capture a wide variety of patterns in the dataset.

I would try random forest, but I find it hard for forecasting weather as it deals with time.

For this model, I want to use **XGBoost** due to its gradient boosting and its overfitting prevention. Also because it is the most interesting model to me.

### Data Collection

I will try to pull data from the Data.gov.hk website and see if I can get data such as how much rainfall, the highest and lowest temperature, humidity, highest wind speed, the visibility, and the amount of sunshine for each day and try to get like, idk, 5 years worth of data?

Website used:
- Daily Mean Pressure: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-mean-pressure
- Daily Total Rainfall: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-total-rainfall
- Daily Max, Mean, Min Temperatures: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-temperature-info-hko
- Daily Mean Relative Humidity: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-mean-relative-humidity
- Daily Mean Amount of Clouds: https://data.gov.hk/en-data/dataset/hk-hko-rss-daily-mean-amount-of-cloud

The most recent date in each file is 5/31/2025.

A total of 7 csv files (Temperature has 3). Let's dig in!

### Libraries

In [95]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

### Data Cleaning/Wrangling
A few things to clarify so you don't have to open the CSVs yourself:

**At the top**, it has this message:

日平均雲量(百分比) - 天文台

Daily Mean Amount of Cloud (%) at the Hong Kong Observatory

年/Year,月/Month,日/Day,數值/Value,數據完整性/data Completeness

So I need to get rid of these rows

**At the bottom**, it has this message:

\- "*** 沒有數據/unavailable"

"# 數據不完整/data incomplete"

"C 數據完整/data Complete"

So I need to get rid of these and adjust accordingly, too.

Sorry this markdown chunk looks really disgusting, I am still not the best with markdown right now I need to learn that too.

In [96]:
def clean_data(dat):
    dat.columns = ['Year', 'Month', 'Day', 'Value', 'Data Complete']
    dat = dat[:-3]
    print("-----Before cleaning-----")

    datDefects = len(dat[ (dat['Value']=="***") | (dat['Value']=="#") ])
    print("Number of faulty rows:",datDefects)
    print("\n---Summary of Value---")
    print(dat['Value'].describe())
    print("\n---Summary of Data Complete---")
    print(dat['Data Complete'].describe())
    
    # Missing data
    dat['Data Complete'].replace(['#', '***'], np.nan, inplace=True)
    dat = dat.dropna()

    dat['Year'] = dat['Year'].astype(int).astype(str)
    dat['Month'] = dat['Month'].astype(int).astype(str)
    dat['Day'] = dat['Day'].astype(int).astype(str)

    dat['Date'] = pd.to_datetime(dat['Year']+"-"+dat['Month']+"-"+dat['Day'], format="%Y-%m-%d")

    dat.drop(['Year', 'Month', 'Day'], axis=1, inplace=True)

    dat = dat.loc[:, ['Date', 'Value', 'Data Complete']]

    print("\n\n-----After cleaning-----")
    print(f"Number of faulty rows: {len(dat[ (dat['Value']=="***") | (dat['Value']=="#") ])}")
    print("\n---Summary of Value---")
    print(dat['Value'].describe())
    print("\n---Summary of Data Complete---")
    print(dat['Data Complete'].describe())
    print("\n---Dataframe Info---")
    print(dat.info())
    return dat

Cleaning HKO_cloud_amount.csv

In [97]:
clouds = pd.read_csv("data/HKO_cloud_amount.csv", header=2)
clouds = clean_data(clouds)

-----Before cleaning-----
Number of faulty rows: 0

---Summary of Value---
count    27910.000000
mean        67.019205
std         25.812034
min          0.000000
25%         50.000000
50%         75.000000
75%         88.000000
max        100.000000
Name: Value, dtype: float64

---Summary of Data Complete---
count     27910
unique        1
top           C
freq      27910
Name: Data Complete, dtype: object


-----After cleaning-----
Number of faulty rows: 0

---Summary of Value---
count    27910.000000
mean        67.019205
std         25.812034
min          0.000000
25%         50.000000
50%         75.000000
75%         88.000000
max        100.000000
Name: Value, dtype: float64

---Summary of Data Complete---
count     27910
unique        1
top           C
freq      27910
Name: Data Complete, dtype: object

---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27910 entries, 0 to 27909
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype        

C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dat['Data Complete'].replace(['#', '***'], np.nan, inplace=True)
C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat['Data Complete'].replace(['#', '***'], np.nan, i

Cleaning HKO_max_temp.csv

In [98]:
maxTemp = pd.read_csv("data/HKO_max_temp.csv", header=2)
maxTemp = clean_data(maxTemp)

-----Before cleaning-----
Number of faulty rows: 1

---Summary of Value---
count     49095
unique      291
top        30.6
freq        501
Name: Value, dtype: object

---Summary of Data Complete---
count     49094
unique        1
top           C
freq      49094
Name: Data Complete, dtype: object


C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dat['Data Complete'].replace(['#', '***'], np.nan, inplace=True)
C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat['Data Complete'].replace(['#', '***'], np.nan, i



-----After cleaning-----
Number of faulty rows: 0

---Summary of Value---
count     49094
unique      290
top        30.6
freq        501
Name: Value, dtype: object

---Summary of Data Complete---
count     49094
unique        1
top           C
freq      49094
Name: Data Complete, dtype: object

---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
Index: 49094 entries, 0 to 49094
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           49094 non-null  datetime64[ns]
 1   Value          49094 non-null  object        
 2   Data Complete  49094 non-null  object        
dtypes: datetime64[ns](1), object(2)
memory usage: 1.5+ MB
None


Cleaning HKO_mean_temp.csv

In [99]:
meanTemp = pd.read_csv("data/HKO_mean_temp.csv", header=2)
meanTemp = clean_data(meanTemp)

-----Before cleaning-----
Number of faulty rows: 32

---Summary of Value---
count     49035
unique      268
top        28.2
freq        595
Name: Value, dtype: object

---Summary of Data Complete---
count     49003
unique        1
top           C
freq      49003
Name: Data Complete, dtype: object


-----After cleaning-----
Number of faulty rows: 0

---Summary of Value---
count     49003
unique      267
top        28.2
freq        595
Name: Value, dtype: object

---Summary of Data Complete---
count     49003
unique        1
top           C
freq      49003
Name: Data Complete, dtype: object

---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
Index: 49003 entries, 31 to 49034
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           49003 non-null  datetime64[ns]
 1   Value          49003 non-null  object        
 2   Data Complete  49003 non-null  object        
dtypes: datetime64[ns](

C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dat['Data Complete'].replace(['#', '***'], np.nan, inplace=True)
C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat['Data Complete'].replace(['#', '***'], np.nan, i

Cleaning HKO_min_temp.csv

In [100]:
minTemp = pd.read_csv("data/HKO_min_temp.csv", header=2)
minTemp = clean_data(minTemp)

-----Before cleaning-----
Number of faulty rows: 1

---Summary of Value---
count     49095
unique      271
top        25.6
freq        654
Name: Value, dtype: object

---Summary of Data Complete---
count     49094
unique        1
top           C
freq      49094
Name: Data Complete, dtype: object


C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dat['Data Complete'].replace(['#', '***'], np.nan, inplace=True)
C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat['Data Complete'].replace(['#', '***'], np.nan, i



-----After cleaning-----
Number of faulty rows: 0

---Summary of Value---
count     49094
unique      270
top        25.6
freq        654
Name: Value, dtype: object

---Summary of Data Complete---
count     49094
unique        1
top           C
freq      49094
Name: Data Complete, dtype: object

---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
Index: 49094 entries, 0 to 49094
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           49094 non-null  datetime64[ns]
 1   Value          49094 non-null  object        
 2   Data Complete  49094 non-null  object        
dtypes: datetime64[ns](1), object(2)
memory usage: 1.5+ MB
None


Cleaning HKO_pressure.csv

In [101]:
pressure = pd.read_csv("data/HKO_pressure.csv", header=2)
pressure = clean_data(pressure)

-----Before cleaning-----
Number of faulty rows: 32

---Summary of Value---
count      49035
unique       417
top       1008.6
freq         323
Name: Value, dtype: object

---Summary of Data Complete---
count     49003
unique        1
top           C
freq      49003
Name: Data Complete, dtype: object


-----After cleaning-----
Number of faulty rows: 0

---Summary of Value---
count      49003
unique       416
top       1008.6
freq         323
Name: Value, dtype: object

---Summary of Data Complete---
count     49003
unique        1
top           C
freq      49003
Name: Data Complete, dtype: object

---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
Index: 49003 entries, 31 to 49034
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           49003 non-null  datetime64[ns]
 1   Value          49003 non-null  object        
 2   Data Complete  49003 non-null  object        
dtypes: datetim

C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dat['Data Complete'].replace(['#', '***'], np.nan, inplace=True)
C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat['Data Complete'].replace(['#', '***'], np.nan, i

Cleaning HKO_rainfall.csv

In [102]:
rainfall = pd.read_csv("data/HKO_pressure.csv", header=2)
rainfall = clean_data(rainfall)

-----Before cleaning-----
Number of faulty rows: 32

---Summary of Value---
count      49035
unique       417
top       1008.6
freq         323
Name: Value, dtype: object

---Summary of Data Complete---
count     49003
unique        1
top           C
freq      49003
Name: Data Complete, dtype: object


-----After cleaning-----
Number of faulty rows: 0

---Summary of Value---
count      49003
unique       416
top       1008.6
freq         323
Name: Value, dtype: object

---Summary of Data Complete---
count     49003
unique        1
top           C
freq      49003
Name: Data Complete, dtype: object

---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
Index: 49003 entries, 31 to 49034
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           49003 non-null  datetime64[ns]
 1   Value          49003 non-null  object        
 2   Data Complete  49003 non-null  object        
dtypes: datetim

C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dat['Data Complete'].replace(['#', '***'], np.nan, inplace=True)
C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat['Data Complete'].replace(['#', '***'], np.nan, i

Cleaning HKO_relative_humidity.csv

In [103]:
humidity = pd.read_csv("data/HKO_relative_humidity.csv", header=2)
humidity = clean_data(humidity)

C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dat['Data Complete'].replace(['#', '***'], np.nan, inplace=True)
C:\Users\danch\AppData\Local\Temp\ipykernel_34460\3100955047.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat['Data Complete'].replace(['#', '***'], np.nan, i

-----Before cleaning-----
Number of faulty rows: 0

---Summary of Value---
count    28641.000000
mean        78.098495
std         11.028292
min         21.000000
25%         73.000000
50%         80.000000
75%         86.000000
max         99.000000
Name: Value, dtype: float64

---Summary of Data Complete---
count     28641
unique        2
top           C
freq      28640
Name: Data Complete, dtype: object


-----After cleaning-----
Number of faulty rows: 0

---Summary of Value---
count    28640.000000
mean        78.098010
std         11.028179
min         21.000000
25%         73.000000
50%         80.000000
75%         86.000000
max         99.000000
Name: Value, dtype: float64

---Summary of Data Complete---
count     28640
unique        1
top           C
freq      28640
Name: Data Complete, dtype: object

---Dataframe Info---
<class 'pandas.core.frame.DataFrame'>
Index: 28640 entries, 0 to 28640
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype         
---